# PHÂN LOẠI VĂN BẢN ESG TIẾNG VIỆT BẰNG HỌC BÁN GIÁM SÁT LẶP KẾT HỢP ĐỒNG THUẬN ĐA MÔ HÌNH TRANSFORMER (ITERATIVE ENSEMBLE PSEUDO-LABELING)

**Loại tài liệu:** Báo cáo Tổng hợp Thực nghiệm Khoa học (Hoàn chỉnh trọn vẹn `B1` → `B8`)  
**Tác giả / Thực hiện:** Lưu Tuấn Trình — Dự án Cuối khóa Xử lý Ngôn ngữ Tự nhiên  
**Trạng thái cập nhật hiện tại:**  **Hoàn tất 100% & thẩm định toàn bộ Bước B1 → B8** (Môi trường, Kiểm toán & EDA, Ablation Baseline, 3 Vòng Iterative Ensemble Pseudo-Labeling, Final Ensemble + Kiểm định McNemar, Hiệu chuẩn ECE, Phân tích Lỗi & Demo tương tác — **27 biểu đồ 300 DPI**)  

---

## Tóm tắt (Abstract)

Phân loại tự động văn bản doanh nghiệp tiếng Việt theo bốn hạng mục **ESG** (*Irrelevant, Environment, Social, Governance*) đóng vai trò then chốt trong định lượng phát triển bền vững, nhưng thường đối mặt với hai rào cản cố hữu: **(i) chi phí gán nhãn chuyên gia cao** khiến dữ liệu không nhãn dồi dào nhưng dữ liệu có nhãn hạn chế, và **(ii) mất cân bằng lớp kết hợp dịch chuyển xác suất tiên nghiệm (Prior Probability Shift)** giữa tập huấn luyện và tập kiểm thử thực tế. Nghiên cứu này thiết kế và chuẩn hóa một quy trình thực nghiệm đảm bảo **không rò rỉ dữ liệu tuyệt đối (Zero Data Leakage)**, kết hợp **Iterative Ensemble Pseudo-Labeling (3 vòng)** trên bảy kiến trúc Transformer đa dạng (4 mô hình tiếng Việt và 3 mô hình đa ngôn ngữ/tiếng Anh) cùng **Ablation Study đối chứng**. Sau khi kiểm toán sạch `84.856` mẫu gốc (`27.365` huấn luyện sạch, `3.000` kiểm thử khóa kín, `54.487` không nhãn sạch, `0` rò rỉ chéo) và khắc phục triệt để lỗi nạp trọng số `float16` trên `Transformers 5.24.0`, hai vòng đồng thuận đa mô hình đã khai thác an toàn `32.903 / 54.487` mẫu không nhãn sạch (`60.4%`, riêng Vòng 2 đạt `97.52%` đồng thuận tuyệt đối 7/7 mô hình), xây dựng tập huấn luyện cuối `39.248` mẫu và kéo **Imbalance Ratio ($IR$) từ `2.29` → `1.61` → `1.35`** khớp sát phân phối kiểm thử (`1.42`). Kết quả trên **Hold-out Test Set (`3.000` mẫu)** chứng minh phương pháp vượt trội toàn diện so với huấn luyện có giám sát thuần túy: ở Vòng 3 (`Run 3`), cả `7/7` mô hình đơn lẻ đều bứt phá (mọi mô hình Baseline đều bị vượt qua: `PhoBERT` `+1.67%`, `Vi-ELECTRA` `+1.50%`, `ViSoBERT` `+4.56%` Macro-F1), với **`DistilBERT-m`** dẫn đầu nhóm đơn lẻ đạt **`89.70%` Accuracy / `89.03%` Macro-F1** ở độ trễ siêu nhanh **`0.324 ms/câu`** (`3.085` mẫu/giây). Hệ thống **Final Ensemble (`Equal Soft 7M`)** thiết lập đỉnh cao mới **`90.77%` Accuracy / `90.09%` Macro-F1 / `90.67%` Weighted-F1**, vượt trội có ý nghĩa thống kê cả Best Single Model (`+1.06%` Macro-F1, kiểm định McNemar $\chi^2 = 5.79, p = 0.0161 < 0.05$) lẫn Best Supervised Baseline (`+3.63%` Macro-F1, $\chi^2 = 32.95, p < 10^{-6}$), đồng thời giảm sai số hiệu chuẩn xác suất (**ECE**) hơn **3.6–4.9 lần** xuống còn **`0.0226`**. Phân tích lỗi trên `277` mẫu sai (`9.23%`) làm rõ `82.31%` lỗi tập trung ở 4 cặp biên (`Irrelevant ↔ Governance` và `Social ↔ Governance`), chủ yếu bắt nguồn từ nhiễu nhãn gốc trên tập kiểm thử, bản chất đa nhãn giao thoa và từ vựng báo cáo tài chính thuần túy.

**Từ khóa:** Phân loại văn bản ESG tiếng Việt, Học bán giám sát (Semi-Supervised Learning), Gán nhãn giả lặp (Iterative Pseudo-Labeling), Học kết hợp (Ensemble Learning), Hiệu chuẩn xác suất (Calibration/ECE), Kiểm định McNemar, Xử lý mất cân bằng dữ liệu.

---
## 1. Giới thiệu & Đặt vấn đề (Introduction)

Trong bối cảnh tài chính bền vững và yêu cầu công bố thông tin phát triển bền vững ngày càng chặt chẽ, việc phân loại tự động các câu/đoạn văn trong báo cáo thường niên và tin tức doanh nghiệp tiếng Việt thành 4 nhóm **Không liên quan (`Irrelevant` — lớp 0)**, **Môi trường (`Environment` — lớp 1)**, **Xã hội (`Social` — lớp 2)** và **Quản trị (`Governance` — lớp 3)** mang lại giá trị thực tiễn cao.

Khác với cách tiếp cận fine-tune đơn lẻ thông thường, nghiên cứu này tái thiết kế và chuẩn hóa phương pháp **Vi-ESG-Bert** theo chuẩn mực thực nghiệm học thuật nghiêm ngặt với 5 đóng góp phương pháp luận cốt lõi:
1. **Kiểm soát Zero Data Leakage tuyệt đối:** Tách tập kiểm định (`Validation`, 10% phân tầng) hoàn toàn từ tập huấn luyện sạch để chọn checkpoint và dừng sớm (*Early Stopping*), khóa kín tập `test.csv` (`3.000` mẫu) chỉ để chấm điểm đánh giá khách quan cuối mỗi vòng.
2. **Đồng nhất hóa tiền xử lý theo kiến trúc (Architecture-Aligned Preprocessing):** Chuẩn hóa Unicode NFC và chỉ kích hoạt tách từ tiếng Việt (`pyvi.ViTokenizer`) cho mô hình được huấn luyện trước ở mức từ (`vinai/phobert-base`), giữ nguyên văn bản tự nhiên cho 6 mô hình subword/SentencePiece; áp dụng nhất quán 100% ở mọi pha huấn luyện, kiểm định, gán nhãn giả và suy luận.
3. **True Unsupervised Ensemble Pseudo-Labeling kết hợp Cân bằng lớp có chọn lọc:** Khai thác `54.487` câu không nhãn thật sự qua 2 vòng bỏ phiếu đồng thuận đa mô hình (Hard Consensus + Weighted Soft Confidence) và chọn lọc theo độ tin cậy giảm dần để khắc phục đồng thời bài toán thiếu dữ liệu và mất cân bằng lớp.
4. **Nghiên cứu Đối chứng (Ablation Study):** Bổ sung mốc huấn luyện có giám sát trên 100% tập `train_pool` gốc (không dùng nhãn giả) nhằm bóc tách định lượng phần hiệu năng thực sự do Pseudo-Labeling mang lại so với việc đơn thuần tăng quy mô dữ liệu gốc.
5. **Đánh giá Toàn diện hướng Triển khai:** Kết hợp đánh giá Final Ensemble trên tập kiểm thử, phân tích hiệu chuẩn xác suất (Calibration/ECE), phân tích lỗi sai tự tin cao (*Error Analysis*) và đo đạc đánh đổi Độ chính xác – Độ trễ (*Accuracy–Latency Trade-off*).

---
## 2. Thiết lập Môi trường Thực nghiệm & Cấu hình Hệ thống (Tổng hợp từ Bước B1)

### 2.1. Hạ tầng phần cứng & phần mềm đã kiểm chứng

Kết quả thực thi kiểm thử hệ thống tại [B1_Kiem_tra_Moi_truong_va_GPU.ipynb](B1_Kiem_tra_Moi_truong_va_GPU.ipynb) xác nhận toàn bộ môi trường tính toán đạt trạng thái ổn định 100% trong môi trường ảo cô lập `.venv`:

| Hạng mục | Cấu hình thực tế ghi nhận tại Bước B1 | Đánh giá kỹ thuật |
| :--- | :--- | :--- |
| **Hệ điều hành & CPU** | Windows 11 (`10.0.26200`), **20 luồng CPU logic** | Đủ năng lực tiền xử lý song song và EDA nhanh |
| **GPU tăng tốc** | **NVIDIA GeForce RTX 5070** — **11.94 GB VRAM**, Compute Capability **`12.0`** (Blackwell) | Hỗ trợ phần cứng **native `torch.bfloat16`**, đã vượt qua phép kiểm thử nhân ma trận BF16 trên CUDA |
| **Nền tảng Python / PyTorch** | Python `3.14.7`, **PyTorch `2.11.0+cu128`** (CUDA `12.8`) | Tương thích trọn vẹn với kiến trúc Blackwell `sm_120` |
| **Hệ sinh thái NLP & ML** | `transformers 5.24.0`, `accelerate 1.16.0`, `datasets 5.2.0`, `safetensors 0.8.0`, `sentencepiece 0.2.3`, `pyvi`, `scikit-learn 1.9.1`, `scipy 1.18.1` | Đầy đủ 17/17 thư viện lõi, không có xung đột phụ thuộc |
| **Trực quan hóa & Tương tác** | `matplotlib 3.11.2`, `seaborn 0.13.2`, `wordcloud 1.9.6`, `ipywidgets 8.1.9` | Đáp ứng xuất hình 300 DPI (`.png` + `.pdf`) và Demo trực tiếp trên JupyterLab |

### 2.2. Chính sách vận hành an toàn phần cứng & danh mục 7 mô hình

Để đảm bảo quá trình huấn luyện kéo dài nhiều vòng không gây quá tải nhiệt độ/VRAM cho GPU 12 GB trên Windows JupyterLab, cấu hình mặc định **`hardware_mode = "balanced"`** được cố định như sau:
- **Độ chính xác hỗn hợp:** `bf16 = True` (ưu việt hơn `fp16` do dải động số mũ rộng ngang `fp32`, giúp ngăn hiện tượng tràn số `NaN` thường gặp ở kiến trúc DeBERTa-v2 như `videberta-base`).
- **Kích thước lô hiệu dụng:** `train_batch_size = 16` kết hợp `gradient_accumulation_steps = 2` (tương đương Effective Batch Size = `32`), duy trì VRAM thực tế trong vùng an toàn ~5–7 GB / 11.94 GB.
- **Ổn định Windows & Tản nhiệt:** `dataloader_num_workers = 0` (tránh lỗi `spawn`/phình RAM đa tiến trình của Windows), nghỉ `cooldown_seconds = 15` và giải phóng bộ nhớ (`gc.collect()`, `torch.cuda.empty_cache()`) sau mỗi mô hình, kết hợp công tắc `skip_if_done = True`.
- **Chỉ số tối ưu mô hình:** `metric_for_best_model = "f1_macro"`, `learning_rate = 2e-5`, `max_length = 256`, `seed = 42`.

| STT | Tên rút gọn | Checkpoint HuggingFace | Họ kiến trúc | Tách từ `pyvi` | Vai trò trong Ensemble |
| :-: | :--- | :--- | :--- | :---: | :--- |
| 1 | `phobert-base` | `vinai/phobert-base` | RoBERTa (VN) | **Có** | Mạnh về ngữ pháp & từ ghép tiếng Việt chuẩn |
| 2 | `electra-base-vn` | `NlpHUST/electra-base-vn` | ELECTRA (VN) | Không | Cơ chế Replaced Token Detection hiệu quả cao về ngữ nghĩa |
| 3 | `visobert` | `uitnlp/visobert` | XLM-R (VN) | Không | Mạnh trên văn bản mạng xã hội & tin tức tiếng Việt hiện đại |
| 4 | `videberta-base` | `Fsoft-AIC/videberta-base` | DeBERTa-v2 (VN) | Không | Disentangled Attention nhạy với quan hệ vị trí trong câu dài |
| 5 | `bert-base-multilingual-cased` | `google-bert/bert-base-multilingual-cased` | mBERT | Không | Bổ sung đa dạng biểu diễn đa ngôn ngữ & thuật ngữ quốc tế |
| 6 | `distilbert-base-multilingual-cased` | `distilbert/distilbert-base-multilingual-cased` | DistilBERT | Không | Nhẹ, suy luận nhanh, làm mốc so sánh Trade-off tốc độ |
| 7 | `roberta-base` | `FacebookAI/roberta-base` | RoBERTa | Không | Đối chứng đa dạng hóa và nhận diện thuật ngữ ESG tiếng Anh xen kẽ |

> **Lưu ý kỹ thuật quan trọng rút ra từ Bước B1:** Phiên bản `transformers 5.24.0` có thay đổi API đáng kể so với dòng v4 cũ (ví dụ sử dụng `eval_strategy` thay cho `evaluation_strategy`, `processing_class` thay cho `tokenizer` trong `Trainer`). Toàn bộ module huấn luyện ở Bước B3 (`src/trainer.py`) sẽ được viết tương thích trực tiếp với **Transformers v5.x**.

---
## 3. Kiểm toán Dữ liệu & Phân tích Khám phá (Data Audit & EDA — Tổng hợp từ Bước B2)

### 3.1. Kiểm toán chất lượng và xác minh Không Rò rỉ Dữ liệu (Zero Data Leakage)

Toàn bộ 3 nguồn dữ liệu gốc (`84.856` dòng) đã được kiểm toán tự động tại [B2_EDA_Lam_sach_va_Chia_Du_lieu.ipynb](B2_EDA_Lam_sach_va_Chia_Du_lieu.ipynb) bằng khóa chuẩn hóa bỏ dấu câu/chữ hoa sau khi quy chuẩn bảng mã tiếng Việt về **Unicode NFC** (`B2_data_audit_report.json`):

| Tập dữ liệu | Số dòng gốc | Rỗng / `< 5` từ | Mâu thuẫn nhãn | Trùng nội bộ | Rò rỉ với Test | Trùng với Train | Số dòng sạch giữ lại | Tỉ lệ bảo toàn |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Train (`train.csv`)** | `27.366` | `0` | `0` | `1` | **`0`** | — | **`27.365`** | `99.996%` |
| **Hold-out Test (`test.csv`)** | `3.000` | `0` | — | `0` | — | **`0`** | **`3.000`** | `100.00%` |
| **Unlabeled (`overall_data.csv`)** | `54.490` | `0` | — | `3` | **`0`** | **`0`** | **`54.487`** | `99.994%` |

![Hình 1: Báo cáo kiểm toán làm sạch và chống rò rỉ dữ liệu](../workspace_outputs/figures/B2_eda/B2_05_kiem_toan_lam_sach_du_lieu.png)

**Đánh giá khoa học (Mục 3.1):**
- Bộ dữ liệu gốc đã được thu thập và sơ chế rất sạch về mặt cấu trúc: không có giá trị khuyết (`NaN`), không có câu rác ngắn dưới 5 từ, và không có mẫu mâu thuẫn nhãn (*label noise* do gán 2 nhãn khác nhau cho cùng 1 câu).
- Quan trọng nhất về phương diện thẩm định khoa học: **không tồn tại bất kỳ sự rò rỉ trùng khớp nào (`0` mẫu)** giữa `test.csv` với `train.csv` và `overall_data.csv`, cũng như giữa `train.csv` và `overall_data.csv`. Do đó, mọi mức cải thiện độ chính xác đo được trên `test_clean.csv` qua các vòng Pseudo-Labeling ở các bước sau đều phản ánh **năng lực tổng quát hóa thực chất**, không bị thổi phồng do rò rỉ dữ liệu.

### 3.2. Phân bố nhãn ESG & Phát hiện Dịch chuyển Xác suất Tiên nghiệm (Prior Probability Shift)

Từ `27.365` mẫu huấn luyện sạch, hệ thống thực hiện chia phân tầng (**Stratified 90/10**, `seed=42`) thu được `train_pool.csv` (`24.628` mẫu) và `val.csv` (`2.737` mẫu), đồng thời trích mẫu ngẫu nhiên 7 tập con `train_subset1..7.csv` (`10.000` mẫu/tập) cho Run 1.

| Lớp ESG | Train Pool (`90%`) | Validation (`10%`) | Hold-out Test (`100%`) | Độ lệch tỉ lệ `(Test − Train)` |
| :--- | :---: | :---: | :---: | :---: |
| **0 — Irrelevant** | `5.312` (`21.57%`) | `590` (`21.56%`) | `600` (`20.00%`) | `-1.57%` |
| **1 — Environment** | `3.945` (`16.02%`) | `439` (`16.04%`) | `820` (`27.33%`) | **`+11.31%`** |
| **2 — Social** | `9.023` (`36.64%`) | `1.003` (`36.65%`) | `850` (`28.33%`) | **`-8.31%`** |
| **3 — Governance** | `6.348` (`25.78%`) | `705` (`25.76%`) | `730` (`24.33%`) | `-1.45%` |
| **Tổng cộng** | **`24.628`** | **`2.737`** | **`3.000`** | — |
| **Tỉ số mất cân bằng ($IR = \max / \min$)** | **`2.29`** | **`2.28`** | **`1.42`** | — |

![Hình 2: So sánh phân bố 4 lớp ESG trên Train Pool, Validation và Hold-out Test](../workspace_outputs/figures/B2_eda/B2_01_phan_bo_lop_esg.png)

**Đánh giá khoa học (Mục 3.2):**
1. **Tính bảo toàn phân tầng hoàn hảo:** Tỉ lệ 4 lớp giữa `train_pool` và `val` khớp chính xác đến 0.02 điểm phần trăm, đảm bảo tập `val.csv` đại diện trung thực cho miền phân bố huấn luyện.
2. **Mất cân bằng lớp rõ rệt trên tập huấn luyện ($IR = 2.29$):** Lớp *Social* (`36.64%`) lớn gấp **2.29 lần** lớp thiểu số *Environment* (`16.02%`). Nếu huấn luyện tối ưu thuần theo `Accuracy`, mô hình có xu hướng thiên lệch dự đoán về lớp đa số *Social*.
3. **Phát hiện Prior Probability Shift ($P_{\text{train}}(y) \neq P_{\text{test}}(y)$):** Tập `test.csv` được thiết kế gần cân bằng ($IR = 1.42$), trong đó tỷ trọng lớp *Environment* tăng vọt thêm **`+11.31` điểm phần trăm** (`16.02%` → `27.33%`) còn *Social* giảm `-8.31%`. Phát hiện này cung cấp **cơ sở lý thuyết trực tiếp** cho thiết kế của nghiên cứu:
   - Bắt buộc chọn checkpoint tốt nhất dựa trên **`Macro-F1`** (trọng số đồng đều giữa 4 lớp) thay vì `Accuracy`.
   - Chiến lược **Selective Class-Balanced Pseudo-Labeling** (ở Bước B4 và B6) không chỉ gia tăng quy mô mẫu mà còn đóng vai trò **hiệu chỉnh phân phối tiên nghiệm (prior alignment)** bằng cách bổ sung tỷ trọng lớn hơn cho các lớp thiếu hụt (*Environment*, *Irrelevant*, *Governance*) so với *Social*.

### 3.3. Phân bố Độ dài Văn bản & Dịch chuyển Độ dài giữa các Miền dữ liệu

| Tập dữ liệu / Lớp ESG | Số mẫu | Trung bình ± Độ lệch chuẩn (từ) | Trung vị (từ) | Phân vị 95% (từ) | Phân vị 99% (từ) | Cực đại (từ) | Tỉ lệ $\le 256$ từ |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Train Pool (Tổng)** | `24.628` | `37.94 ± 18.71` | `35.0` | `67.0` | `89.0` | `766` | `99.97%` |
| ↳ Lớp 0: *Irrelevant* | `5.312` | `26.9` | `25.0` | `47.0` | — | — | — |
| ↳ Lớp 1: *Environment* | `3.945` | `38.0` | `36.0` | `65.0` | — | — | — |
| ↳ Lớp 2: *Social* | `9.023` | `40.2` | `38.0` | `68.0` | — | — | — |
| ↳ Lớp 3: *Governance* | `6.348` | `43.9` | `42.0` | `74.6` | — | — | — |
| **Validation (Tổng)** | `2.737` | `37.98 ± 17.55` | `35.0` | `66.2` | `95.6` | `254` | `100.00%` |
| **Hold-out Test (Tổng)** | `3.000` | **`44.27 ± 25.77`** | **`39.0`** | **`84.0`** | **`133.0`** | `454` | `99.90%` |
| **Unlabeled (Tổng)** | `54.487` | **`48.28 ± 33.81`** | **`41.0`** | **`97.0`** | **`170.0`** | `796` | `99.60%` |

![Hình 3: Phân bố độ dài câu và cơ sở chọn max_length = 256 token](../workspace_outputs/figures/B2_eda/B2_02_phan_bo_do_dai_van_ban.png)

**Đánh giá khoa học (Mục 3.3):**
1. **Cấu trúc độ dài phân tầng theo độ phức tạp ngữ nghĩa:** Boxplot bên phải cho thấy độ dài trung vị tăng đơn điệu giữa 4 lớp: `Irrelevant (25)` < `Environment (36)` < `Social (38)` < `Governance (42)`. Hiện tượng này hoàn toàn phù hợp về mặt ngôn ngữ học miền tài chính: các phát ngôn *Quản trị* thường mô tả cơ cấu hội đồng quản trị, quy chế tuân thủ và quản lý rủi ro dưới dạng câu ghép dài, trong khi câu *Irrelevant* thường là tiêu đề, ghi chú kỹ thuật hoặc nhận định ngắn gọn. (Tuy nhiên, cần lưu ý mô hình không nên lệ thuộc quá mức vào độ dài vì trên `test.csv`, lớp `Irrelevant` có trung bình `37.7` từ — dài hơn đáng kể so với `26.9` từ ở tập `train_pool`!)
2. **Phát hiện Covariate Length Shift (Dịch chuyển độ dài câu):** Cả tập `test_clean` (`Mean = 44.27`, `P99 = 133`) và tập `unlabeled_clean` (`Mean = 48.28`, `P99 = 170`) đều có đuôi phân bố dài hơn đáng kể so với `train_pool` (`Mean = 37.94`, `P99 = 89` — thể hiện rõ bằng đường cong KDE màu cam dịch sang phải và có một cụm đuôi tại ngưỡng cắt `120` từ ở Hình 3). Điều này càng củng cố lợi ích của Pseudo-Labeling: việc đưa thêm các câu dài sạch từ `overall_data.csv` vào quá trình huấn luyện Vòng 2–3 sẽ giúp các mô hình thích nghi tốt hơn với cấu trúc câu dài của tập `test.csv`!
3. **Chứng minh lựa chọn `max_length = 256` token:** Với tỷ lệ mở rộng subword trung bình ~1.3–1.5 token/từ trong tiếng Việt, ngưỡng `256` token bao phủ trọn vẹn trên **99%** số câu của cả `train_pool` (`P99 = 89` từ ≈ `125` token), `test_clean` (`P99 = 133` từ ≈ `190` token) và xấp xỉ `99%` tập `unlabeled` (`P99 = 170` từ ≈ `240` token), đạt điểm cân bằng tối ưu giữa bảo toàn ngữ cảnh và chi phí tính toán chú ý $O(L^2)$ trên GPU 12 GB.

### 3.4. Phân tích Từ vựng Đặc trưng, Độ tách biệt Ngữ nghĩa & Dự báo Vùng Khó

![Hình 4: Top-12 từ/cụm từ đặc trưng nhất của từng lớp ESG theo điểm TF-IDF](../workspace_outputs/figures/B2_eda/B2_03_tu_khoa_dac_trung_4_lop.png)

![Hình 5: Mây từ vựng (WordCloud) theo 4 nhóm chủ đề ESG](../workspace_outputs/figures/B2_eda/B2_04_wordcloud_4_lop_esg.png)

**Đánh giá khoa học (Mục 3.4):**
1. **Độ tách biệt từ vựng rõ rệt giữa 3 trụ cột E–S–G:**
   - **Lớp 1 (`Environment`):** Có biên độ điểm đặc trưng TF-IDF cao vượt trội (lên tới `> 0.064`), hội tụ chặt vào hệ thuật ngữ chuyên ngành hẹp (*phát thải, khí thải, chất thải, năng lượng, nhà kính, nước, tái chế, biến đổi khí hậu*). **Giả thuyết thực nghiệm 1:** Lớp *Environment* sẽ đạt F1-Score cao nhất (thường trên 95%) dù có số lượng mẫu gốc ít nhất.
   - **Lớp 2 (`Social`):** Hội tụ rõ quanh nhóm chủ thể con người và phúc lợi (*nhân viên, khách hàng, cộng đồng, cung cấp, hỗ trợ, lao động, an toàn, sức khỏe, đào tạo*).
   - **Lớp 3 (`Governance`):** Hội tụ quanh cơ chế điều hành và giám sát (*quản lý, quản trị, giám sát, rủi ro, tuân thủ, báo cáo, kiểm toán, chính sách, chiến lược, ủy ban*).
2. **Nhận diện vùng nhập nhằng và thách thức chính (`Irrelevant` & biên `Social–Governance`):**
   - **Lớp 0 (`Irrelevant`)** có biên độ điểm đặc trưng TF-IDF rất thấp (`< 0.015`, chỉ bằng ~1/4 so với *Environment*), từ vựng phân tán vào các khái niệm chung của báo cáo doanh nghiệp (*kinh doanh, chương trình, cung cấp, báo cáo, bền vững*). Kết hợp với sự dịch chuyển độ dài câu của lớp 0 (`26.9` từ ở Train lên `37.7` từ ở Test), **Giả thuyết thực nghiệm 2** được đặt ra: *Irrelevant* sẽ là lớp khó phân loại nhất (F1 thấp nhất ở Run 1) và sẽ hưởng lợi nhiều nhất từ quá trình bổ sung dữ liệu đa dạng ở Vòng 1–2 Pseudo-Labeling.
   - Tồn tại vùng giao thoa từ vựng tự nhiên giữa `Social` và `Governance` (cùng dùng các cụm *chính sách, tổ chức, doanh nghiệp, chương trình, phát triển bền vững*), đòi hỏi mô hình Transformer hiểu ngữ cảnh sâu thay vì chỉ dựa vào từ khóa.
3. **Nhận xét phản biện kỹ thuật về trực quan hóa (Self-Critique):**
   - Toàn bộ 5 biểu đồ lưu tại `workspace_outputs/figures/B2_eda/` đều hiển thị tiếng Việt có dấu rõ nét, không vỡ chữ, đủ cả `.png` (300 DPI) và `.pdf` vector.
   - *Điểm có thể tinh chỉnh thêm nếu cần hoàn thiện tối đa cho bản in luận văn:* Ở Hình 4 & Hình 5 vẫn còn một số từ chức năng/âm tiết đơn tách từ từ ghép tiếng Việt (*"ta", "câu", "thành", "sử dụng", "cách"*) do danh sách stopword cơ bản chưa bao phủ hết và tách token theo khoảng trắng chưa qua `ViTokenizer`; ngoài ra, một vài từ cỡ lớn trong WordCloud dùng dải màu sáng (`Greys`/`Blues`/`Oranges`) có độ tương phản nhẹ trên nền trắng. Các hình này vẫn hoàn toàn đạt yêu cầu thuyết trình, và có thể nâng cấp nhanh bảng stopword + dải màu đậm nếu tác giả yêu cầu ở bước rà soát cuối.

### 3.5. Kết luận Giai đoạn B1–B2 & Sẵn sàng chuyển sang Bước B3

| Tiêu chí kiểm định B1–B2 | Trạng thái | Ghi chú chuyển tiếp sang Bước B3 |
| :--- | :---: | :--- |
| Hạ tầng GPU RTX 5070 + PyTorch `cu128` + `BF16` | ✅ Đạt 100% | Dùng chế độ `balanced` (`bs=16 × accum=2`, `bf16=True`) |
| Kiểm toán Zero Data Leakage (`train`/`val`/`test`/`unlabeled`) | ✅ Đạt 100% | `0` câu rò rỉ chéo; đã xuất đủ CSV sạch tại `processed_data/` |
| Chia tập `train_pool` (24.628) / `val` (2.737) / `7 subsets` (10.000) | ✅ Đạt 100% | Sẵn sàng train cả **Ablation Baseline** lẫn **Run 1** |
| Bộ 5 biểu đồ EDA chuẩn 300 DPI (`PNG + PDF`) | ✅ Đạt 100% | Đã lưu tại `workspace_outputs/figures/B2_eda/` & nhúng vào báo cáo |

**Kết luận đánh giá:** Giai đoạn `B1–B2` đã hoàn thành xuất sắc với chất lượng dữ liệu sạch, không rò rỉ và đã khám phá được 3 bằng chứng định lượng đắt giá (**Imbalance $IR=2.29$**, **Prior Shift lớp Environment $+11.31\%$**, **Covariate Length Shift từ $37.9 \to 48.3$ từ**) làm nền tảng lý luận vững chắc cho toàn bộ thực nghiệm `B3–B8`.

---
## 4. Kết quả Thực nghiệm Đối chứng (Ablation Baseline) & Vòng 1 (Tổng hợp từ Bước B3)

### 4.1. Bảng tổng hợp hiệu năng trên Hold-out Test Set (`3.000` mẫu khóa kín)

Kết quả thực thi tại [B3_Train_Baseline_va_Run1.ipynb](B3_Train_Baseline_va_Run1.ipynb) (`B3_baseline_and_run1_summary.csv`) cho thấy quy trình huấn luyện trên RTX 5070 vận hành rất ổn định và mát máy (mức VRAM đỉnh chỉ `2.95–5.86 GB / 11.94 GB`, thời gian mỗi mô hình Run 1 chỉ `1.7–6.0` phút):

| Vòng chạy | Mô hình | Best Epoch | Val Macro-F1 | **Test Acc** | **Test Macro-F1** | Test Weighted-F1 | F1 Irrelevant (0) | F1 Environment (1) | F1 Social (2) | F1 Governance (3) | ECE ↓ | VRAM (GB) | Thời gian |
| :--- | :--- | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| **Baseline** *(24.6k)* | **PhoBERT-base** | 8 | `90.38%` | `87.80%` | **`86.46%`** | `87.44%` | `73.29%` | **`97.30%`** | `88.45%` | `86.80%` | `0.1106` | `3.82` | `7.2m` |
| **Baseline** *(24.6k)* | **Vi-ELECTRA** | 5 | `90.06%` | `87.07%` | `85.81%` | `86.77%` | `73.53%` | `97.15%` | `88.02%` | `84.54%` | `0.1157` | `3.82` | `6.8m` |
| **Baseline** *(24.6k)* | **ViSoBERT** | 7 | `86.65%` | `84.73%` | `82.93%` | `84.06%` | `67.28%` | `94.31%` | `85.70%` | `84.44%` | `0.1356` | `3.29` | `8.9m` |
| **Run 1** *(10k sub5)* | **Vi-ELECTRA** | 3 | `88.03%` | **`88.03%`** | **`87.29%`** | **`88.05%`** | **`77.49%`** | `96.50%` | `88.72%` | `86.45%` | **`0.0576`** | `3.81` | `2.1m` |
| **Run 1** *(10k sub1)* | **PhoBERT-base** | 7 | `89.57%` | `87.67%` | `86.43%` | `87.38%` | `73.43%` | `97.17%` | `87.94%` | **`87.17%`** | `0.1061` | `3.84` | `3.1m` |
| **Run 1** *(10k sub3)* | **mBERT-cased** | 5 | `88.10%` | `87.60%` | `86.06%` | `87.15%` | `71.89%` | `96.73%` | **`89.94%`** | `85.70%` | `0.0596` | `4.46` | `2.4m` |
| **Run 1** *(10k sub2)* | **DistilBERT-m** | 4 | `86.11%` | `85.37%` | `83.86%` | `84.90%` | `70.20%` | `95.98%` | `86.01%` | `83.23%` | `0.0686` | `2.95` | `1.7m` |
| **Run 1** *(10k sub6)* | **ViSoBERT** | 5 | `84.54%` | `82.90%` | `80.37%` | `81.79%` | `60.54%` | `92.94%` | `85.13%` | `82.85%` | `0.1243` | `3.33` | `3.5m` |
| **Run 1** *(10k sub4)* | **RoBERTa-base** | 6 | `80.95%` | `81.33%` | `79.28%` | `80.56%` | `61.52%` | `92.05%` | `82.28%` | `81.28%` | `0.0882` | `3.70` | `6.0m` |
| **Run 1** *(10k sub7)* | **ViDeBERTa-base** ⚠️ | 1 | `13.41%` | `28.33%` | `11.04%` | `12.51%` | `0.00%` | `0.00%` | `44.16%` | `0.00%` | `0.1494` | `5.86` | `2.0m` |

![Hình 6: Đường cong huấn luyện Vòng 1 (Run 1 — 7 Subsets)](../workspace_outputs/figures/B3_baseline_run1/B3_01_learning_curves_run1.png)

![Hình 7: Đường cong huấn luyện Đối chứng (Ablation Baseline — 100% Train Pool)](../workspace_outputs/figures/B3_baseline_run1/B3_01b_learning_curves_baseline.png)

![Hình 8: So sánh Hiệu năng trên Hold-out Test Set — Baseline vs Run 1](../workspace_outputs/figures/B3_baseline_run1/B3_02_so_sanh_hieu_nang_run1.png)

### 4.2. Phân tích Khoa học & Kiểm chứng Giả thuyết từ Bước B2

1. **Kiểm chứng thành công 100% cả hai giả thuyết đặt ra từ phân tích EDA (Mục 3.4):**
   - **Giả thuyết 1 (Khẳng định):** Dù lớp `Environment` chỉ chiếm `16.02%` dữ liệu huấn luyện (thấp nhất trong 4 lớp), mọi mô hình hội tụ đều đạt F1 cao vượt trội ở lớp này (`92.05%` – `97.30%`), đúng như dự báo từ biên độ điểm TF-IDF đặc trưng rất cao (`> 0.064`).
   - **Giả thuyết 2 (Khẳng định):** Lớp `Irrelevant` là điểm nghẽn chính của toàn hệ thống với F1 thấp nhất trên cả 6 mô hình hội tụ (`60.54%` – `77.49%`), thấp hơn lớp `Environment` tới **19–32 điểm phần trăm**.
2. **Phát hiện chuyên sâu về Hiệu chuẩn Xác suất & Giới hạn của Huấn luyện có giám sát thuần túy (Baseline vs Run 1):**
   - Một kết quả đáng chú ý cấp luận văn là **Vi-ELECTRA Run 1** (dừng sớm ở Epoch 3 trên `10k` mẫu) đạt **Test Macro-F1 = `87.29%`** và **`ECE = 0.0576`**, vượt trội hơn chính **Vi-ELECTRA Baseline** (`24.6k` mẫu, Epoch 5: Test Macro-F1 = `85.81%`, `ECE = 0.1157`), mặc dù trên tập `val.csv` bản Baseline cao hơn (`90.06%` vs `88.03%`).
   - Quan sát Hình 7 (*Learning Curves Baseline*) giải thích rõ cơ chế này (*Guo et al., 2017 — On Calibration of Modern Neural Networks*): từ Epoch 3 trở đi, `Train Loss` giảm sâu về `< 0.10` trong khi **`Val Loss` tăng dần trở lại (`0.35 → 0.70`)** dù `Val Macro-F1` vẫn nhích nhẹ. Mô hình trở nên **quá tự tin (over-confident)** (`ECE` tăng gấp đôi lên `0.11–0.135`) và khớp quá chặt vào phân bố lệch của `train_pool` ($IR=2.29$), dẫn tới suy giảm khả năng tổng quát hóa sang phân bố cân bằng hơn của `Hold-out Test` ($IR=1.42$). Kết quả đối chứng này chứng minh: **chỉ tăng lượng dữ liệu gốc mất cân bằng là không đủ**; cần bổ sung dữ liệu cân bằng hóa phân phối 4 lớp qua Pseudo-Labeling.
3. **Chẩn đoán chính xác nguyên nhân suy biến (`Mode Collapse`) của `ViDeBERTa-base` trên `Transformers v5.24.0` & Khắc phục triệt để:**
   - Trong Hình 6 và `run1_videberta-base_history.csv`, `ViDeBERTa-base` giữ nguyên `train_loss ≈ 2.69–2.74` qua cả 3 epoch và dự đoán 100% lớp đa số `Social` (`Test Acc = 28.33% = 850/3000`, `Macro-F1 = 11.04%`), ngay cả khi tắt `BF16`.
   - **Nguyên nhân gốc rễ đã kiểm chứng thực nghiệm:** Kho checkpoint `Fsoft-AIC/videberta-base` trên HuggingFace lưu sẵn cấu hình `"torch_dtype": "float16"`. Khác với dòng `transformers v4` (mặc định nạp trọng số gốc ở `float32`), phiên bản **`transformers 5.24.0`** mặc định nạp trực tiếp tham số mô hình theo `config.json` thành **trọng số gốc thuần `torch.float16`**! Khi tối ưu bằng `AdamW` với `lr = 1.5e-5` trên trọng số gốc `float16` (không có bản sao master weights FP32), độ lớn bước cập nhật nhỏ hơn độ phân giải dấu phẩy động `float16` (~`6e-5`), khiến toàn bộ trọng số bị **đóng băng (underflow về 0)**.
   - **Khắc phục đã kiểm thử thành công 100% (`src/model_wrapper.py`):** Truyền tường minh **`dtype=torch.float32`** trong `AutoModelForSequenceClassification.from_pretrained(...)` để luôn duy trì *FP32 master weights* cho tất cả mô hình (kiểm thử trực tiếp cho thấy loss của `ViDeBERTa-base` giảm ngay từ bước cập nhật đầu tiên: `1.4828 → 1.4157`), đồng thời giữ `force_fp32=True`, `lr=1.5e-5` cho DeBERTa-v2 và tích hợp **Quality Gate (`min_model_val_f1 = 0.50`)** vào `EnsembleVoter`.

---
## 5. Phân tích Chất lượng Pseudo-Labeling & Ensemble Voting (Vòng 1 — Bước B4)

### 5.1. Suy luận trên `54.487` mẫu Không nhãn, Đồng thuận Đa mô hình & Cân bằng lớp Vòng 1

Ở Bước B4 ([B4_PseudoLabel_va_Ensemble_Vong1.ipynb](B4_PseudoLabel_va_Ensemble_Vong1.ipynb)), 7 mô hình Run 1 thực hiện suy luận độc lập trên `54.487` câu sạch trong `unlabeled_clean.csv` với tốc độ ấn tượng trên RTX 5070 (`1.037 – 3.725` mẫu/giây):

| Mô hình (Run 1) | Trọng số `Val F1` | Số mẫu đạt `Conf ≥ 0.94` | Tỉ lệ vượt ngưỡng | Mean Conf | Mean Entropy | Tốc độ (mẫu/s) |
| :--- | :-: | :-: | :-: | :-: | :-: | :-: |
| **PhoBERT-base** | `0.1688` | `49.081` | `90.08%` | `0.9729` | `0.0780` | `2.717,5` |
| **mBERT-cased** | `0.1660` | `36.839` | `67.61%` | `0.9104` | `0.2572` | `2.301,9` |
| **Vi-ELECTRA** | `0.1659` | `37.783` | `69.34%` | `0.9143` | `0.2719` | `2.378,6` |
| **DistilBERT-m** | `0.1623` | `34.150` | `62.68%` | `0.8967` | `0.3026` | **`3.724,9`** |
| **ViSoBERT** | `0.1593` | `42.923` | `78.78%` | `0.9423` | `0.1583` | `1.820,3` |
| **RoBERTa-base** | `0.1525` | `29.639` | `54.40%` | `0.8717` | `0.3538` | `1.036,9` |
| **ViDeBERTa-base** ⚠️ | `0.0253` | `0` | `0.00%` | `0.4328` | `1.2921` | `1.324,3` |

![Hình 9: Phân bố độ tin cậy (Confidence) của 7 mô hình Run 1 trên 54.487 câu không nhãn](../workspace_outputs/figures/B4_pseudo_round1/B4_01_phan_bo_confidence_vong1.png)

![Hình 10: Ma trận đồng thuận cặp (%) và hệ số Cohen's Kappa Vòng 1](../workspace_outputs/figures/B4_pseudo_round1/B4_02_ma_tran_dong_thuan_kappa.png)

![Hình 11: Phân bố mức độ đồng thuận (Ensemble Consensus Voting Vòng 1)](../workspace_outputs/figures/B4_pseudo_round1/B4_03_phan_bo_so_phieu_bau.png)

![Hình 12: Hiệu chỉnh mất cân bằng lớp và Prior Shift sau Combine Vòng 1](../workspace_outputs/figures/B4_pseudo_round1/B4_04_can_bang_lop_sau_vong1.png)

**Đánh giá khoa học (Vòng 1 — Bước B4):**
1. **Bằng chứng thực nghiệm về tính bền vững của Weighted Soft Voting:** Nhờ cơ chế gán trọng số tỷ lệ theo `Val Macro-F1`, mô hình `ViDeBERTa-base` bị collapse tự động bị hạ trọng số xuống chỉ còn `2.53%` (`0.0253`), nên không làm ô nhiễm chất lượng xác suất của tập nhãn giả lọc ra (**`23.378 / 54.487` mẫu = `42.91%`**, đạt chất lượng rất cao với **`Avg Confidence = 0.9682`**, **`Avg Margin = 0.9533`**, **`Avg Entropy = 0.1722`**).
2. **Phân tích Ma trận Cohen's Kappa (Hình 10):** Nếu loại riêng hàng/cột `ViDeBERTa-base` (`Kappa = 0.000`), sáu mô hình hội tụ đạt mức đồng thuận rất lành mạnh (**`73.5% – 83.4%`**, **Cohen's Kappa `0.635 – 0.774`**, trung bình **`0.706`** — thuộc ngưỡng *Substantial Agreement* theo thang Landis & Koch). Cặp đồng thuận cao nhất là `PhoBERT-base ↔ Vi-ELECTRA` (`83.4%`, `κ = 0.774`) và `DistilBERT-m ↔ mBERT-cased` (`83.4%`, `κ = 0.771`), trong khi sự khác biệt giữa nhóm tiếng Việt và `RoBERTa-base` (`κ ≈ 0.64`) tạo ra độ đa dạng (*ensemble diversity*) quý giá cho bỏ phiếu.
3. **Hiệu quả hiệu chỉnh Prior Shift & Điểm nghẽn cần xử lý tiếp (Hình 12):**
   - Ba trụ cột E–S–G đều đạt trọn vẹn mục tiêu cân bằng `9.500` mẫu/lớp: **Environment** tăng mạnh từ `3.945` lên **`9.500`** (`+5.555` mẫu sạch), **Governance** tăng từ `6.348` lên **`9.500`** (`+3.152`), **Social** từ `9.023` lên **`9.500`** (`+477`). Tổng tập huấn luyện tăng từ `24.628` lên **`34.404` mẫu**, giúp giảm mạnh **Imbalance Ratio ($IR$) từ `2.29` xuống `1.61`**!
   - **Điểm cần lưu ý ở lớp 0 (`Irrelevant`):** Do ngưỡng đồng thuận cao (`conf ≥ 0.94`) cộng hưởng với việc `ViDeBERTa-base` luôn bầu cho `Social`, lớp `Irrelevant` thu được **`592`** mẫu (`5.312 → 5.904`).
   - **Định hướng tối ưu cho Vòng 2 (`B5 → B6`):** Nhờ bản vá **`dtype=torch.float32`** trong `src/model_wrapper.py`, từ **Bước B5 (`Run 2`)** trở đi cả **7/7 mô hình** đều nạp master weights ở FP32 và hội tụ đầy đủ, kết hợp **Quality Gate** và ngưỡng hạ nhẹ (`τ = 0.91`) trên `31.109` mẫu dư (`remain_data_r1.csv`) ở Bước B6 sẽ bổ sung mạnh lượng mẫu chất lượng cao cho lớp `Irrelevant`.

### 5.2. Kết quả Pseudo-Labeling & Ensemble Voting Vòng 2 (Bước B6 — Trên `31.109` câu khó còn lại)

Ở Bước B6 ([B6_PseudoLabel_va_Ensemble_Vong2.ipynb](B6_PseudoLabel_va_Ensemble_Vong2.ipynb)), **đủ cả 7/7 mô hình Run 2 hội tụ** tham gia suy luận và bỏ phiếu có trọng số rất cân bằng (`0.1365 – 0.1477`) trên `remain_data_r1.csv` (`31.109` mẫu chưa vượt qua Vòng 1) với ngưỡng `τ = 0.91`, `margin ≥ 0.45`, `vote ≥ 5/7`:

| Mô hình (Run 2) | Trọng số `Val F1` | Số mẫu đạt `Conf ≥ 0.91` | Tỉ lệ vượt ngưỡng | Mean Conf | Median Conf | Mean Entropy | Tốc độ (mẫu/s) |
| :--- | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| **Vi-ELECTRA** | **`0.1477`** | `23.630` | `75.96%` | `0.9237` | `0.9888` | `0.2089` | `2.217,7` |
| **mBERT-cased** | `0.1463` | `29.267` | `94.08%` | `0.9817` | `0.9999` | `0.0465` | `2.130,1` |
| **PhoBERT-base** | `0.1458` | `20.787` | `66.82%` | `0.8950` | `0.9702` | `0.2810` | `2.546,5` |
| **DistilBERT-m** | `0.1443` | `24.688` | `79.36%` | `0.9350` | `0.9947` | `0.1715` | **`3.388,7`** |
| **ViSoBERT** | `0.1416` | `28.025` | `90.09%` | `0.9696` | `0.9998` | `0.0765` | `1.797,0` |
| **RoBERTa-base** | `0.1378` | `21.240` | `68.28%` | `0.9003` | `0.9770` | `0.2562` | `1.032,2` |
| **ViDeBERTa-base** ✅ | `0.1365` | `16.957` | `54.51%` | `0.8555` | `0.9304` | `0.3741` | `1.227,7` |

![Hình 13: Biểu đồ phễu khai thác dữ liệu không nhãn (Data Funnel qua 2 vòng)](../workspace_outputs/figures/B6_pseudo_round2/B6_01_bieu_do_pheu_unlabeled.png)

![Hình 14: Phân bố tập huấn luyện cuối cùng — Stacked Bar 3 tầng (Original + Pseudo 1 + Pseudo 2)](../workspace_outputs/figures/B6_pseudo_round2/B6_02_phan_bo_du_lieu_3_tang.png)

![Hình 15: Đường cong tích lũy ngược độ tin cậy (1 - ECDF) của 7 mô hình Run 2 trên remain_data_r1](../workspace_outputs/figures/B6_pseudo_round2/B6_03_phan_bo_confidence_vong2.png)

![Hình 16: Ma trận đồng thuận cặp (%) và hệ số Cohen's Kappa Vòng 2 (7/7 mô hình hội tụ)](../workspace_outputs/figures/B6_pseudo_round2/B6_04_ma_tran_dong_thuan_kappa_vong2.png)

**Đánh giá khoa học (Vòng 2 — Bước B6):**
1. **Độ tinh khiết cực cao của nhãn giả Vòng 2:** Trên `31.109` câu khó còn lại, Ensemble Vòng 2 thu hoạch thêm **`9.525` mẫu (`30.62%`)**, trong đó có tới **`9.289 / 9.525` mẫu (`97.52%`) đạt đồng thuận TUYỆT ĐỐI 7/7 mô hình** (`Avg Confidence = 0.9738`, `Avg Margin = 0.9522`, `Avg Entropy = 0.1205`). Tính gộp cả 2 vòng (*Hình 13*), hệ thống khai thác an toàn **`32.903 / 54.487` mẫu (`60.4%`)** và chủ động loại bỏ **`21.584` mẫu (`39.6%`)** có độ bất định cao để tránh nhiễu nhãn.
2. **Cải tiến trực quan hóa từ KDE sang đường cong Tích lũy Ngược (`1 - ECDF`, Hình 9 & Hình 15):** Do các mô hình Transformer sau fine-tune có phân bố độ tin cậy lệch trái cực mạnh (`median_conf = 0.930 – 0.9999`), mật độ KDE truyền thống dựng đỉnh rất cao sát `1.0` (`y ≈ 40`) làm ép phẳng vùng `[0.40, 0.90]`. Chuyển sang biểu diễn **Survival Curve (`P(Confidence ≥ x)`)** cho thấy tách bạch rõ từng đường cong của cả 7 mô hình: `mBERT-cased` và `ViSoBERT` tự tin cao nhất (`>90%` đạt `≥ 0.91`), trong khi `ViDeBERTa-base` (`54.5%`) và `PhoBERT-base` (`66.8%`) bảo thủ và hiệu chuẩn xác suất tốt nhất (`ECE` thấp nhất) — đóng vai trò "bộ phanh chống quá tự tin" quý giá cho tập thể Ensemble!
3. **Hoàn tất hiệu chỉnh Prior Shift ($P_{\text{train}}(y) \approx P_{\text{test}}(y)$, Hình 14):**
   - Vòng 2 đã giải quyết dứt điểm điểm nghẽn lớp `Irrelevant` bằng cách bổ sung thêm **`+1.844` mẫu** (từ `5.904 → 7.748`), đồng thời đưa 3 lớp E–S–G lên tròn **`10.500`** mẫu/lớp, tạo thành tập huấn luyện cuối cùng **`combined_round2.csv` (`39.248` mẫu)**.
   - **Đối chiếu phân phối cuối cùng với Hold-out Test (`3.000` mẫu):** `Irrelevant` = **`19.74%`** (Test `20.00%`), `Environment` = **`26.75%`** (Test `27.33%`), `Social` = **`26.75%`** (Test `28.33%`), `Governance` = **`26.75%`** (Test `24.33%`). **Imbalance Ratio ($IR$)** giảm liên tục qua 3 mốc: **`2.29` (Gốc) → `1.61` (Sau Vòng 1) → `1.35` (Sau Vòng 2)** — khớp gần như tuyệt đối với phân phối kiểm thử thực tế (`1.42`).

---
## 6. Diễn tiến Huấn luyện Vòng 2–3 & Đánh giá Final Ensemble

### 6.1. Kết quả Huấn luyện Vòng 2 (`Run 2` trên `combined_round1.csv` — Bước B5)

Kết quả thực thi tại [B5_Train_Run2.ipynb](B5_Train_Run2.ipynb) (`B5_run2_summary.csv`) ghi nhận **cả 7/7 mô hình đều hội tụ đầy đủ** với mức VRAM đỉnh siêu an toàn (`2.96 – 4.47 GB / 11.94 GB`):

| Mô hình | Best Epoch | Val Macro-F1 | **Test Acc (R2)** | **Test Macro-F1 (R2)** | Δ Macro-F1 (R2−R1) | F1 Irrel (0) | F1 Env (1) | F1 Soc (2) | F1 Gov (3) | ECE ↓ | VRAM (GB) | Thời gian |
| :--- | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| **mBERT-cased** | 8 | `90.07%` | **`88.00%`** | **`86.92%`** | `+0.86%` | `76.35%` | `96.94%` | `87.97%` | **`86.42%`** | `0.1123` | `4.47` | `24.5m` |
| **Vi-ELECTRA** | 3 | **`90.93%`** | `87.77%` | `86.79%` | `-0.50%` | `76.29%` | **`97.65%`** | `87.12%` | `86.12%` | `0.0911` | `3.82` | `6.8m` |
| **PhoBERT-base** | 3 | `89.78%` | `87.27%` | `85.89%` | `-0.54%` | `71.89%` | `97.40%` | **`88.93%`** | `85.33%` | `0.0881` | `3.84` | `6.5m` |
| **DistilBERT-m** | 5 | `88.86%` | `87.10%` | `85.80%` | **`+1.94%`** | `73.47%` | `96.27%` | `87.37%` | `86.10%` | `0.1029` | `2.96` | `6.3m` |
| **ViSoBERT** | 7 | `87.22%` | `85.77%` | `84.98%` | **`+4.61%`** | **`76.70%`** | `94.74%` | `82.85%` | `85.64%` | `0.1295` | `3.30` | `13.2m` |
| **RoBERTa-base** | 8 | `84.88%` | `86.00%` | `84.49%` | **`+5.21%`** | `71.26%` | `94.46%` | `87.76%` | `84.49%` | `0.1015` | `3.70` | `19.0m` |
| **ViDeBERTa-base** ✅ | 7 | `84.03%` | `84.90%` | `83.75%` | **`+72.71%`** | `72.62%` | `93.56%` | `84.34%` | `84.46%` | **`0.0853`** | `3.17` | `87.5m` |

![Hình 17: Đường cong huấn luyện Vòng 2 (Run 2 — 7/7 mô hình hội tụ trên combined_round1.csv)](../workspace_outputs/figures/B5_run2/B5_01_learning_curves_run2.png)

![Hình 18: Bước nhảy hiệu năng trên Hold-out Test Set (Run 1 → Run 2)](../workspace_outputs/figures/B5_run2/B5_02_buoc_nhay_run1_sang_run2.png)

**Đánh giá khoa học (Vòng 2 — Bước B5):**
1. **Kiểm chứng thực nghiệm bản vá `ViDeBERTa-base`:** Kết hợp `dtype=torch.float32` master weights + `BF16` AMP + `bs=8 × accum=4` + `activation_checkpointing=True` đã giải cứu hoàn toàn `ViDeBERTa-base` từ `11.04%` (collapse) lên **`83.75%` Macro-F1 / `84.90%` Acc**, đạt độ hiệu chuẩn xác suất tốt nhất hệ thống (**`ECE = 0.0853`**) với mức VRAM chỉ **`3.17 GB`**!
2. **Mức bứt phá mạnh ở nhóm mô hình yếu & đa ngôn ngữ:** `RoBERTa-base` (`+5.21%` Macro-F1), `ViSoBERT` (`+4.61%` Macro-F1; đặc biệt lớp `Irrelevant` tăng vọt **`+16.16%`** từ `60.54% → 76.70%` và vượt cả bản `ViSoBERT Baseline` `82.93% → 84.98%`!), `DistilBERT-m` (`+1.94%`) và `mBERT-cased` (`+0.86%`, vươn lên dẫn đầu `88.00%` Acc). Nhìn chung, mặt bằng chất lượng của cả 7 thành viên Ensemble thu hẹp lại rất đều quanh `83.75% – 86.92%` Macro-F1.
3. **Giải thích cơ chế chững nhẹ ở `Vi-ELECTRA` (`-0.50%`) và `PhoBERT-base` (`-0.54%`) tại Run 2:** Trên `val.csv`, `Vi-ELECTRA` thực tế tăng mạnh từ `88.03% → 90.93%` (`+2.90%`, vượt cả bản Baseline `90.06%`) và cả hai đều lập kỷ lục mới ở lớp `Environment` (`97.65%` và `97.40%`). Tuy nhiên, do ở Vòng 1 tỷ trọng lớp `Irrelevant` trong `combined_round1.csv` tạm thời bị co xuống còn `17.2%` (chỉ có `5.904` mẫu so với `9.500` của 3 lớp kia), F1 lớp `Irrelevant` trên Test giảm nhẹ `1.2–1.5%`. Nút thắt này đã được tháo gỡ hoàn toàn tại Bước B6 (`combined_round2.csv` nâng lớp `Irrelevant` lên `7.748` mẫu = `19.74%`), tạo tiền đề vững chắc để cả 7 mô hình và **Final Ensemble** bứt phá ở **Bước B7 (`Run 3`)**.

### 6.2. Kết quả Huấn luyện Vòng 3 (`Run 3`), Final Ensemble & Kiểm định Thống kê (Tổng hợp từ Bước B7)

Tại Bước B7 ([B7_Train_Run3_va_Danh_gia_Final_Ensemble.ipynb](B7_Train_Run3_va_Danh_gia_Final_Ensemble.ipynb)), toàn bộ 7 kiến trúc Transformer được khởi tạo sạch từ trọng số gốc (*fresh pretrained*) và huấn luyện trên tập cuối cùng `combined_round2.csv` (`39.248` mẫu, $IR = 1.35$), sau đó đánh giá tổng hợp 4 chiến lược **Final Ensemble** và đo tốc độ suy luận chuẩn hóa trên RTX 5070 (`B7_run3_and_ensemble_summary.csv`, `run3_final_ensemble_eval.json`):

| Nhóm / Mô hình | Best Ep | Val Macro-F1 | **Test Acc** | **Test Macro-F1** | Δ F1 (R3−R2) | F1 Irrel (0) | F1 Env (1) | F1 Soc (2) | F1 Gov (3) | **ECE ↓** | Latency (ms/câu) | Throughput (mẫu/s) | Params | VRAM / Thời gian |
| :--- | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| **DistilBERT-m** *(Best Single)* | 4 | `89.37%` | **`89.70%`** | **`89.03%`** | **`+3.23%`** | **`79.87%`** | `97.77%` | `88.21%` | **`90.26%`** | **`0.0825`** | **`0.324`** | **`3.085,3`** | `135.3M` | `2.96 GB` / `6.3m` |
| **mBERT-cased** | 6 | `89.82%` | `89.43%` | `88.48%` | `+1.56%` | `78.92%` | `96.89%` | **`89.54%`** | `88.58%` | `0.0988` | `0.550` | `1.817,9` | `177.9M` | `4.47 GB` / `14.5m` |
| **PhoBERT-base** | 5 | **`91.10%`** | `88.97%` | `88.13%` | `+2.24%` | `78.41%` | **`97.88%`** | `88.16%` | `88.08%` | `0.1018` | `0.493` | `2.028,0` | `135.0M` | `3.86 GB` / `10.5m` |
| **ViSoBERT** | 8 | `87.64%` | `88.30%` | `87.49%` | `+2.51%` | `78.69%` | `97.09%` | `87.98%` | `86.22%` | `0.1117` | `0.601` | `1.662,6` | `97.6M` | `3.30 GB` / `19.8m` |
| **Vi-ELECTRA** | 4 | `90.55%` | `88.23%` | `87.31%` | `+0.52%` | `77.41%` | `97.47%` | `87.43%` | `86.92%` | `0.1055` | `0.495` | `2.018,8` | `133.7M` | `3.83 GB` / `9.8m` |
| **RoBERTa-base** | 6 | `85.26%` | `88.13%` | `86.95%` | `+2.46%` | `75.61%` | `95.86%` | `89.28%` | `87.04%` | `0.0898` | `0.977` | `1.023,6` | `124.6M` | `3.70 GB` / `22.1m` |
| **ViDeBERTa-base** | 6 | `83.55%` | `86.17%` | `85.14%` | `+1.39%` | `74.38%` | `95.16%` | `85.47%` | `85.56%` | `0.0864` | `0.834` | `1.199,2` | `184.3M` | `3.36 GB` / `32.2m` |
| **Ensemble — Equal Soft (7M)** 🏆 | — | — | **`90.77%`** | **`90.09%`** | — | **`82.54%`** | **`98.18%`** | **`90.39%`** | `89.24%` | **`0.0226`** | `4.274` | `234,0` | `988.4M` | — |
| **Ensemble — Weighted Soft (7M)** | — | — | `90.73%` | `90.04%` | — | `82.36%` | `98.12%` | `90.39%` | **`89.29%`** | `0.0256` | `4.274` | `234,0` | `988.4M` | — |
| **Ensemble — Hard Voting (7M)** | — | — | `90.63%` | `89.95%` | — | `82.36%` | `98.12%` | `90.20%` | `89.11%` | `0.0290` | `4.274` | `234,0` | `988.4M` | — |
| **Ensemble — Top-3 Weighted** *(Pho+Vi+mBERT)* | — | — | `89.90%` | `89.08%` | — | `80.14%` | `98.01%` | `89.52%` | `88.64%` | `0.0589` | `1.538` | `650,2` | `446.5M` | — |

![Hình 19: Đường cong huấn luyện Vòng 3 (Run 3 — 7/7 mô hình hội tụ trên combined_round2.csv)](../workspace_outputs/figures/B7_run3_ensemble/B7_00_learning_curves_run3.png)

![Hình 20: So sánh 7 Mô hình Đơn lẻ (Run 3) vs 4 Chiến lược Final Ensemble trên Hold-out Test Set](../workspace_outputs/figures/B7_run3_ensemble/B7_01_so_sanh_don_le_va_ensemble.png)

![Hình 21: Benchmark Tốc độ Suy luận (Throughput) & Quy mô Tham số của 7 mô hình trên RTX 5070](../workspace_outputs/figures/B7_run3_ensemble/B7_02_benchmark_toc_do_suy_luan.png)

**Kiểm định ý nghĩa thống kê McNemar (McNemar's Test với hiệu chỉnh liên tục Edwards, $\alpha = 0.05$):**

| Cặp so sánh trên Hold-out Test (`3.000` mẫu) | Acc (A vs B) | Số mẫu A sửa đúng (B sai) | Số mẫu A làm sai (B đúng) | Chênh lệch ròng | Thống kê $\chi^2$ | Giá trị $p$ | Kết luận thống kê |
| :--- | :-: | :-: | :-: | :-: | :-: | :-: | :--- |
| **Equal Soft (7M)** vs **Best Single (`DistilBERT-m`)** | `90.77%` vs `89.70%` | **`99`** | `67` | **`+32`** | `5.7892` | **`0.016125`** | ✅ **Có ý nghĩa thống kê ($p < 0.05$)** |
| **Equal Soft (7M)** vs **Best Baseline (`PhoBERT-base`)** | `90.77%` vs `87.80%` | **`162`** | `73` | **`+89`** | `32.9532` | **`< 0.000001`** | ✅ **Ý nghĩa thống kê cực mạnh ($p < 10^{-6}$)** |

**Đánh giá khoa học (Vòng 3 & Final Ensemble — Bước B7):**
1. **Xác nhận dứt điểm giả thuyết cân bằng hóa lớp `Irrelevant` (100% mô hình tăng trưởng đồng loạt):** Sau khi Vòng 2 nâng số mẫu lớp `Irrelevant` lên `7.748` (`19.74%`, khớp `20.00%` của tập Test), **cả 7/7 mô hình đều bật tăng mạnh so với Run 2 (`+0.52%` đến `+3.23%` Macro-F1)** — đặc biệt F1 lớp `Irrelevant` của `PhoBERT-base` bật vọt `+6.52%` (`71.89% → 78.41%`) và `DistilBERT-m` tăng `+6.40%` (`73.47% → 79.87%`). Ngoài ra, việc nâng `batch_size_override = 16` cho `ViDeBERTa-base` giúp rút thời gian huấn luyện giảm **2.72 lần** (`87.45m → 32.16m`) trong khi VRAM vẫn giữ cực mát ở `3.36 GB`.
2. **Phân tích hiện tượng học thuật thú vị — Vì sao `DistilBERT-m` (6 lớp) vượt qua `PhoBERT-base` (12 lớp) trên Test Set (`89.03%` vs `88.13%`) dù `PhoBERT-base` dẫn đầu Validation (`91.10%` vs `89.37%`)?**
   - Khoảng cách tổng quát hóa (`Val − Test` Macro-F1) của `DistilBERT-m` cực hẹp (`+0.34%`), trong khi `PhoBERT-base` (`+2.97%`) và `Vi-ELECTRA` (`+3.24%`) có độ lệch lớn hơn. Nguyên nhân kết hợp 3 yếu tố: **(i)** `DistilBERT-m` dừng sớm ở Epoch 4 với độ hiệu chuẩn tốt nhất nhóm đơn lẻ (`ECE = 0.0825`, ít *over-confident* hơn `PhoBERT` `0.1018` và `Vi-ELECTRA` `0.1055`); **(ii)** kiến trúc chưng cất 6 lớp mang tính điều chuẩn cấu trúc (*inductive regularization*) mạnh khi học trên dữ liệu có trộn nhãn giả và dịch chuyển độ dài câu (`Covariate Length Shift`); và **(iii)** bộ từ vựng đa ngôn ngữ của `DistilBERT-m` & `mBERT-cased` xử lý mượt mà các câu báo cáo ESG tiếng Việt có pha trộn thuật ngữ tiếng Anh/tên tổ chức quốc tế (*GRI, SHEQ, Net Zero, Techmart, Nedbank*) mà không bị phân mảnh bởi bộ tách từ `pyvi`.
3. **Hiệu ứng đa dạng kiến trúc trong Final Ensemble (`Equal Soft 7M` phá mốc `90%` cả Acc & Macro-F1):**
   - Cả 3 biến thể Ensemble 7 mô hình đều bỏ xa toàn bộ 7 mô hình đơn lẻ (`89.95% – 90.09%` Macro-F1). Đáng chú ý, **`Equal Soft (7M)` (`90.09%`)** nhỉnh nhẹ hơn `Weighted Soft (7M)` (`90.04%`) vì trọng số tính theo `Val Macro-F1` vô tình ưu tiên hơn cho `PhoBERT`/`Vi-ELECTRA` (có khoảng cách Val–Test rộng hơn) và hạ nhẹ tỷ trọng của `DistilBERT-m`, `RoBERTa-base` và `ViDeBERTa-base` (những mô hình có độ hiệu chuẩn và độ đa dạng cao hơn trên Test).
   - So sánh **`Top-3 Weighted` (`89.08%`)** với **`Equal Soft 7M` (`90.09%`)** chứng minh định lượng giá trị của việc huy động đủ **7 họ kiến trúc khác nhau**: thêm 4 mô hình còn lại mang lại bước nhảy **`+1.01%` Macro-F1**, **`+2.40%` F1 lớp `Irrelevant`** (`80.14% → 82.54%`) và kéo **ECE giảm hơn 2.6 lần** (`0.0589 → 0.0226`).

---
## 7. Tổng hợp Tiến trình 3 Vòng, Thảo luận Chuyên sâu & Phân tích Lỗi (Tổng hợp từ Bước B8)

### 7.1. Bức tranh toàn cảnh 3 Vòng Pseudo-Labeling & Bằng chứng dứt điểm của Ablation Study

Bảng tổng hợp xuyên suốt từ `Baseline` → `Run 1` → `Run 2` → `Run 3` → `Final Ensemble` (trích xuất từ `summary_all_runs.csv` tại [B8_Truc_quan_hoa_Tong_hop_Phan_tich_Loi_va_Demo.ipynb](B8_Truc_quan_hoa_Tong_hop_Phan_tich_Loi_va_Demo.ipynb)) cung cấp bằng chứng thực nghiệm không thể bác bỏ cho hiệu quả của phương pháp đề xuất:

| Mô hình / Hệ thống | Baseline *(100% Train Pool — 24.6k)* | Run 1 *(10k Subset)* | Run 2 *(34.4k + P1)* | Run 3 *(39.2k + P2)* | Tăng trưởng **(R3 − Baseline)** | Tăng trưởng **(R3 − R1)** |
| :--- | :-: | :-: | :-: | :-: | :-: | :-: |
| **PhoBERT-base** | `86.46%` *(87.80%)* | `86.43%` *(87.67%)* | `85.89%` *(87.27%)* | **`88.13%`** *(88.97%)* | **`+1.67%`** | `+1.70%` |
| **Vi-ELECTRA** | `85.81%` *(87.07%)* | `87.29%` *(88.03%)* | `86.79%` *(87.77%)* | **`87.31%`** *(88.23%)* | **`+1.50%`** | `+0.02%` |
| **ViSoBERT** | `82.93%` *(84.73%)* | `80.37%` *(82.90%)* | `84.98%` *(85.77%)* | **`87.49%`** *(88.30%)* | **`+4.56%`** | **`+7.12%`** |
| **DistilBERT-m** | — | `83.86%` *(85.37%)* | `85.80%` *(87.10%)* | **`89.03%`** *(89.70%)* | — | **`+5.17%`** |
| **mBERT-cased** | — | `86.06%` *(87.60%)* | `86.92%` *(88.00%)* | **`88.48%`** *(89.43%)* | — | `+2.42%` |
| **RoBERTa-base** | — | `79.28%` *(81.33%)* | `84.49%` *(86.00%)* | **`86.95%`** *(88.13%)* | — | **`+7.67%`** |
| **ViDeBERTa-base** | — | `11.04%` *(Collapse)* | `83.75%` *(84.90%)* | **`85.14%`** *(86.17%)* | — | *Hồi sinh hoàn toàn* |
| **Final Ensemble — Equal Soft (7M)** | — | — | — | **`90.09%` *(90.77%)*** | **`+3.63%`** *(vs Best Base)* | **`+2.80%`** *(vs Best R1)* |

> *(Ghi chú bảng trên: Giá trị chính là **Test Macro-F1 (%)**, giá trị trong ngoặc đơn là **Test Accuracy (%)** trên `3.000` mẫu Hold-out Test khóa kín.)*

![Hình 22: Biểu đồ chủ lực — Tiến trình 3 vòng Pseudo-Labeling so với Đối chứng Ablation Baseline & Final Ensemble](../workspace_outputs/figures/B8_final_report/B8_01_tien_trinh_3_vong_va_ablation.png)

![Hình 23: Lưới Biểu đồ Radar 5 trục — Tiến bộ toàn diện qua 3 vòng của 7 mô hình & Tổng kết Baseline vs R3 vs Ensemble](../workspace_outputs/figures/B8_final_report/B8_02_radar_charts_7_models.png)

**Nhận định cốt lõi từ Ablation Study (Hình 22 & Hình 23):**
- **Bóc tách thành công đóng góp thực chất của Pseudo-Labeling:** Trên cả 3 kiến trúc tiếng Việt có đối chứng 100% `train_pool` gốc, việc kết hợp nhãn giả đồng thuận cân bằng lớp giúp vượt xa ngưỡng trần có giám sát thuần túy (`PhoBERT +1.67%`, `Vi-ELECTRA +1.50%`, `ViSoBERT +4.56%`). Nói cách khác, giá trị của quy trình **không đến từ việc mở rộng dữ liệu gốc đơn thuần**, mà đến từ **ba cơ chế cộng hưởng**: *(1) Tri thức phân bố từ `32.903` câu mới giúp mở rộng độ phủ ngữ nghĩa và thích nghi câu dài*, *(2) Cân bằng phân phối lớp $IR: 2.29 \to 1.35$ tháo gỡ Prior Shift*, và *(3) Chưng cất ngầm giữa 7 kiến trúc (Cross-Model Knowledge Transfer)* — giải thích vì sao các mô hình vốn yếu hơn ở Run 1 (`RoBERTa-base +7.67%`, `ViSoBERT +7.12%`, `DistilBERT-m +5.17%`) tăng mạnh nhất khi được học lại từ nhãn giả do cả tập thể cùng đồng thuận sinh ra!
- Quan sát ô Radar thứ 8 ở **Hình 23** (*Baseline vs R3 vs Ensemble*), đa giác của `Best Ensemble` bao trùm hoàn toàn cả `Best Baseline` lẫn `Best Single R3`, với phần diện tích mở rộng rõ rệt nhất nằm đúng ở đỉnh khó nhất **`F1 Irrelevant`** (`73.29%` Baseline → `79.87%` Best Single → **`82.54%`** Ensemble, tăng **`+9.25` điểm phần trăm**!).

### 7.2. Phân tích Chiều sâu theo Lớp ESG, Ma trận Nhầm lẫn, Pareto Trade-off & Hiệu chuẩn Xác suất

![Hình 24: Heatmap F1-Score từng lớp ESG (%) — 7 mô hình Run 3 và 4 chiến lược Final Ensemble](../workspace_outputs/figures/B8_final_report/B8_03_heatmap_f1_theo_lop.png)

![Hình 25: Ma trận Nhầm lẫn (Confusion Matrix) — Best Single Model (DistilBERT-m) vs Best Final Ensemble (Equal Soft 7M)](../workspace_outputs/figures/B8_final_report/B8_04_confusion_matrix_best_va_ensemble.png)

![Hình 26: Đánh đổi Hiệu năng (Macro-F1) vs Độ trễ Suy luận (Latency) & Đường biên tối ưu Pareto](../workspace_outputs/figures/B8_final_report/B8_05_tradeoff_hieu_nang_va_toc_do.png)

![Hình 27: Biểu đồ Hiệu chuẩn Xác suất (Reliability Diagram & Chỉ số ECE) — DistilBERT-m vs Equal Soft Ensemble (7M)](../workspace_outputs/figures/B8_final_report/B8_06_calibration_reliability.png)

**Đánh giá khoa học (Mục 7.2):**
1. **Cơ chế sửa lỗi bổ trợ của Ensemble trên Ma trận Nhầm lẫn (Hình 24 & Hình 25):**
   - So với `DistilBERT-m`, **`Equal Soft (7M)`** cắt giảm hơn **một nửa** số lỗi bỏ sót của hai trụ cột quan trọng: lỗi `Social → Irrelevant` giảm mạnh từ **`91` xuống `39`** (nâng Recall lớp `Social` từ `83.2% → 87.4%` và F1 từ `88.21% → 90.39%`), lỗi `Governance → Irrelevant` giảm từ **`32` xuống `13`** (nâng Recall `Governance` từ `94.5% → 97.1%`). Nhờ lọc bỏ phần lớn dương tính giả rơi vào lớp 0, Precision của `Irrelevant` tăng vọt từ `79.4%` lên **`89.2%`**, đưa `F1 Irrelevant` vượt mốc **`82.54%` (`+2.67%`)**.
   - Lớp **`Environment`** đạt ngưỡng gần tiệm cận hoàn hảo trên toàn hệ thống (`810 / 820` mẫu đúng, `Recall = 98.78%`, `Precision = 97.59%`, **`F1 = 98.18%`**), chỉ sai tổng cộng 10 câu trên toàn bộ 820 mẫu.
2. **Đột phá về Độ tin cậy Xác suất — Calibration / ECE (Hình 27):**
   - Đây là một trong những kết quả giá trị nhất về mặt ứng dụng thực tế: các mô hình đơn lẻ sau fine-tune đều mắc chứng **quá tự tin (over-confidence)** với `ECE = 0.0825 – 0.1117` (ví dụ ở `DistilBERT-m`, các dự đoán rơi vào bin tin cậy `[0.5, 0.6]` thực tế chỉ đúng `30%`, bin `[0.7, 0.8]` chỉ đúng `56%`).
   - Khi kết hợp trung bình xác suất 7 kiến trúc khác biệt (`Equal Soft 7M`), **đường hiệu chuẩn bám sát đường chéo lý tưởng $y = x$ trên mọi khoảng tin cậy**, đưa **`ECE` giảm mạnh hơn 3.6–4.9 lần xuống `0.0226`**. Cụ thể tại bin cao nhất `[0.9, 1.0]`, có tới **`2.316 / 3.000` câu (`77.2%` tập Test)** đạt độ chính xác thực tế **`97.41%`** (so với xác suất trung bình `99.35%`, sai lệch chỉ `1.94%`). Đặc tính hiệu chuẩn tốt này mở đường cho cơ chế **Human-in-the-Loop**: hệ thống có thể tự động phê duyệt các câu có `Confidence ≥ 0.90` (độ chính xác `> 97.4%`) và chỉ chuyển `~22.8%` câu bất định còn lại sang chuyên gia ESG kiểm duyệt!
3. **Khuyến nghị Triển khai theo Đường biên tối ưu Pareto (Hình 26):**
   - **Kịch bản 1 — Thời gian thực / Chi phí hạ tầng tối thiểu (Real-time API / Edge):** Chọn **`DistilBERT-m (Run 3)`** (`89.03%` Macro-F1, `0.324 ms/câu`, **`3.085` mẫu/giây**, VRAM `2.96 GB`) — giữ được **98.8%** sức mạnh của Ensemble 7 mô hình nhưng nhanh hơn **13.2 lần** và gọn nhẹ hơn **7.3 lần** về tham số.
   - **Kịch bản 2 — Cân bằng Sản xuất (Production Balanced):** Chọn **`Top-3 Weighted Ensemble`** (`89.08%` Macro-F1, `89.90%` Acc, `1.54 ms/câu`, `650` mẫu/giây, `ECE = 0.0589`) — tối ưu chi phí vận hành khi cần độ ổn định cao hơn mô hình đơn lẻ.
   - **Kịch bản 3 — Kiểm toán ESG / Độ chính xác & Hiệu chuẩn tối đa (Offline Audit / Batch Processing):** Chọn **`Equal Soft Ensemble (7M)`** (`90.09%` Macro-F1, `90.77%` Acc, `ECE = 0.0226`, `4.27 ms/câu` ≈ `234` mẫu/giây — vẫn đủ sức phân loại trọn vẹn `10.000` câu báo cáo thường niên chỉ trong **~43 giây** trên 1 GPU tầm trung RTX 5070).

### 7.3. Phân tích Lỗi Định lượng & Định tính Chuyên sâu (Error Analysis trên `277 / 3.000` mẫu sai)

Tổng hợp toàn bộ `277` câu dự đoán sai (`9.23%`) của hệ thống tốt nhất `Equal Soft (7M)` từ `B8_confusion_pairs.csv` và `B8_top15_errors.csv`:

| Thứ hạng | Cặp nhầm lẫn `(Nhãn thực → Nhãn đoán)` | Số lượng lỗi | Tỉ trọng trong `277` lỗi | Nhận diện bản chất |
| :-: | :--- | :-: | :-: | :--- |
| **1** | `Irrelevant → Governance` | **`85`** | **`30.69%`** | Giao thoa báo cáo tài chính/kế toán + nhiễu nhãn gốc |
| **2** | `Social → Governance` | **`62`** | **`22.38%`** | Biên đa nhãn tự nhiên (HĐQT họp/quyết định về nhân sự, DEI, an toàn lao động) |
| **3** | `Irrelevant → Social` | **`42`** | **`15.16%`** | Câu mô tả khách hàng/dịch vụ kinh doanh chung chung mang sắc thái xã hội |
| **4** | `Social → Irrelevant` | **`39`** | **`14.08%`** | Câu phúc lợi/hoạt động nội bộ thiếu từ khóa ESG tường minh |
| 5–12 | 8 cặp nhầm lẫn còn lại *(trong đó `Environment` chỉ sai tổng cộng `10` câu)* | `49` | `17.69%` | Phân tán nhỏ lẻ (`Governance → Irrelevant: 13`, `Irrelevant → Environment: 12`, ...) |

**Thẩm định định tính Top-15 câu sai có độ tự tin cao nhất (`Confidence = 99.70% – 99.97%`, `B8_top15_errors.csv`) phát hiện 3 nhóm nguyên nhân gốc rễ mang giá trị phản biện học thuật cao:**

1. **Nhóm A — Nhiễu nhãn gốc trên tập Hold-out Test (Ground-Truth Label Noise):** Một tỷ lệ đáng kể các ca "sai tự tin > 99.8%" thực chất lại cho thấy **mô hình Ensemble suy luận hợp lý hơn nhãn gốc** trong bộ dữ liệu dịch/thu thập thô:
   - *Dòng `2474` (`True: Irrelevant` → `Pred: Environment`, Conf `99.97%`):* *"điểm nổi bật của greco 2024 là khu vực 'techmart' trưng bày **công nghệ xanh, năng lượng tái tạo và các giải pháp môi trường** để chuyển đổi công nghiệp."* — Rõ ràng mang trọn vẹn ngữ nghĩa Môi trường.
   - *Dòng `413` (`True: Irrelevant` → `Pred: Environment`, Conf `99.87%`):* *"**94% vật liệu trong bao bì được làm từ vật liệu tái sử dụng, tái chế, có thể phân hủy** sau khi chế xong."* — Tiêu chuẩn điển hình của kinh tế tuần hoàn (`Environment`).
   - *Dòng `185` (`True: Irrelevant` → `Pred: Governance`, Conf `99.83%`):* *"cô ấy có khả năng lãnh đạo xuất sắc trong **chiến lược công ty, quản lý rủi ro và quản trị công ty**."* & *Dòng `1022` (`Conf 99.93%`)* nói về *"**hệ thống kiểm soát nội bộ hiệu quả** và kiểm soát các quy trình tài chính"* — mang từ khóa lõi của trụ cột Quản trị.
   - **Hàm ý khoa học:** Tập `test.csv` tồn tại một tỷ lệ nhiễu nhãn nền (*annotation noise*), đồng nghĩa hiệu năng thực chất của `Equal Soft (7M)` trên chuẩn chuyên gia ESG còn cao hơn mốc `90.77%` ghi nhận trên nhãn thô.
2. **Nhóm B — Giao thoa Đa nhãn cố hữu tại biên `Social ↔ Governance` (`22.38%` tổng lỗi):**
   - *Dòng `1173` (`True: Social` → `Pred: Governance`, Conf `99.70%`):* *"**hội đồng quản trị** hiện tại của chúng tôi bao gồm **năm phụ nữ, hai người mỹ gốc phi**, một giám đốc đến từ ấn độ..."* — Câu này đồng thời thuộc **Governance** (*Cơ cấu Hội đồng quản trị / Board Composition*) lẫn **Social** (*Đa dạng & Hòa nhập / DEI*).
   - *Dòng `2194` & `2874` (`True: Social` → `Pred: Governance`, Conf `99.87% / 99.79%`):* Mô tả **cuộc họp Hội đồng quản trị** xem xét báo cáo về văn hóa doanh nghiệp, lực lượng lao động và sức khỏe an toàn `SHEQ / COVID-19`. Việc ép bài toán về **Đơn nhãn (Single-Label)** buộc mô hình chọn duy nhất 1 trong 2 trụ cột, tạo ra giới hạn lỗi Bayes (*Bayes Error Floor*) không thể triệt tiêu hoàn toàn nếu giữ nguyên định nghĩa nhãn đơn.
3. **Nhóm C — Giao thoa từ vựng Kế toán / Tài chính thuần túy (`Irrelevant → Governance`, `30.69%` tổng lỗi):**
   - *Dòng `1546`, `1277`, `2694` (`Conf 99.72% – 99.93%`):* Trình bày phương pháp hạch toán chuẩn mực, báo cáo tài chính hợp nhất, khoản vay liên công ty, hiệu lực kỳ báo cáo thường niên. Các câu này chia sẻ dày đặc lớp từ vựng hành chính – pháp lý với `Governance`, lý giải vì sao trên Confusion Matrix lớp `Governance` có `Recall` cực cao (`97.12%`) nhưng `Precision` thấp nhất trong 3 trụ cột (`82.54%`).

---
## 8. Kết luận, Hạn chế & Hướng Phát triển (Conclusion & Future Work)

### 8.1. Kết luận tổng hợp

Nghiên cứu đã xây dựng, triển khai và thẩm định hoàn chỉnh quy trình phân loại văn bản ESG tiếng Việt bằng **Iterative Ensemble Pseudo-Labeling (3 vòng)** trên nền tảng **7 kiến trúc Transformer đa dạng**, hoàn tất trọn vẹn từ kiểm toán dữ liệu (`B1–B2`), huấn luyện đối chứng và bán giám sát lặp (`B3–B7`), cho tới phân tích lỗi và công cụ Demo tương tác (`B8`) với 5 kết quả khẳng định chính:

1. **Tính liêm chính khoa học tuyệt đối (Zero Data Leakage):** Toàn bộ `84.856` mẫu đã được kiểm toán sạch (`0` trùng chéo với `test.csv`), chuẩn hóa tiền xử lý đồng nhất theo kiến trúc (`pyvi` riêng cho `PhoBERT`) và kiểm chứng kỹ thuật cho mọi mốc kết quả.
2. **Hiệu quả thực chứng của Iterative Ensemble Pseudo-Labeling + Cân bằng lớp:** Khai thác an toàn `32.903 / 54.487` câu không nhãn sạch (`60.4%`) qua 2 vòng đồng thuận đa mô hình giúp giảm `Imbalance Ratio` từ `2.29 → 1.35`, tháo gỡ `Prior Shift` & `Covariate Length Shift`, đồng thời vượt xa **Ablation Baseline (100% dữ liệu gốc)** trên mọi kiến trúc kiểm chứng (`+1.50%` đến `+4.56%` Macro-F1).
3. **Hiệu năng đỉnh cao & Có ý nghĩa thống kê:** **`Equal Soft Ensemble (7M)`** đạt **`90.77%` Accuracy / `90.09%` Macro-F1 / `90.67%` Weighted-F1**, vượt trội có ý nghĩa thống kê so với cả mô hình đơn lẻ tốt nhất `DistilBERT-m` (`89.03%` Macro-F1, McNemar $p = 0.0161$) và Baseline mạnh nhất `PhoBERT-base` (`86.46%` Macro-F1, McNemar $p < 10^{-6}$).
4. **Độ tin cậy xác suất sẵn sàng triển khai thực tế:** Ensemble 7 mô hình giảm sai số hiệu chuẩn **`ECE` xuống `0.0226`** (cải thiện hơn `3.6–4.9×` so với từng mô hình đơn lẻ), đạt độ chính xác thực tế **`97.41%`** trên `77.2%` tập kiểm thử có độ tin cậy `≥ 0.90`.
5. **Độ bền kỹ thuật & Tối ưu tài nguyên:** Chẩn đoán và xử lý triệt để lỗi nạp `float16` trên `Transformers v5.24.0` (giải cứu hoàn toàn `ViDeBERTa-base`), giữ đỉnh VRAM của cả 7 mô hình dưới `4.5 GB / 11.94 GB` trên RTX 5070, đồng thời cung cấp rõ 3 điểm triển khai trên đường biên Pareto (`DistilBERT-m 0.324ms` → `Top-3 1.54ms` → `Equal Soft 7M 4.27ms`).

### 8.2. Hạn chế của nghiên cứu

- **Giới hạn định nghĩa đơn nhãn (Single-Label Bottleneck):** `22.38%` tổng số lỗi tập trung ở biên `Social ↔ Governance` do nhiều câu báo cáo ESG thực tế đồng thời đề cập cả quản trị HĐQT lẫn quyền lợi/đa dạng lao động.
- **Nhiễu nhãn nền & bản dịch máy trong kho dữ liệu gốc:** Một số câu trong `test.csv` còn chứa lỗi gán nhãn (`row 2474`, `413`, `185`, `1022`) hoặc câu dịch chưa mượt (*"những vấn đề chúng ta..."*, *"chủ đề nóng lên..."*), tạo trần đánh giá nhân tạo.
- **Chi phí suy luận của Ensemble đầy đủ:** `Equal Soft (7M)` cần nạp `988.4M` tham số (`4.27 ms/câu`), lớn gấp ~7.3 lần về bộ nhớ so với triển khai một mô hình đơn lẻ như `DistilBERT-m`.
- **Phân tích ở mức câu độc lập (Sentence-Level):** Thiếu ngữ cảnh các câu liền kề trong báo cáo thường niên khiến hệ thống đôi khi khó phân biệt thuyết minh báo cáo tài chính thuần túy (`Irrelevant`) với công bố quản trị rủi ro (`Governance`).

### 8.3. Hướng phát triển tương lai

1. **Chưng cất tri thức từ Ensemble sang Mô hình Gọn nhẹ (Ensemble Knowledge Distillation):** Sử dụng phân phối xác suất đã hiệu chuẩn cực tốt (`ECE = 0.0226`) của `Equal Soft (7M)` làm *Teacher* để huấn luyện một mô hình *Student* duy nhất (`DistilBERT-m` hoặc `ViSoBERT`) bằng hàm mất mát KL-Divergence, hướng tới mục tiêu giữ nguyên mốc **`~90%` Macro-F1 ở độ trễ `0.32 ms/câu`**.
2. **Mở rộng sang Phân loại Đa nhãn (Multi-Label ESG) & Phân tầng chi tiết theo chuẩn GRI / SASB:** Cho phép một câu mang đồng thời nhiều nhãn trụ cột (ví dụ `[Social, Governance]`) và đi sâu tới các tiểu mục (*Phát thải khí nhà kính, Bình đẳng giới, Chống tham nhũng*).
3. **Làm sạch nhãn bán tự động bằng Confident Learning (Cleanlab) & Hậu hiệu chỉnh biên quyết định:** Tận dụng chính xác suất của Ensemble 7 mô hình để rà soát nhiễu nhãn trên tập dữ liệu gốc, đồng thời tối ưu ngưỡng xác suất theo từng lớp trên tập Validation nhằm nâng cao `Precision` của cặp `Irrelevant → Governance`.
4. **Phân tích ESG theo Ngữ cảnh Đoạn văn (Paragraph/Document-Level) kết hợp LLM:** Sử dụng mô hình cửa sổ ngữ cảnh dài hoặc kỹ thuật chưng cất suy luận từ các Mô hình Ngôn ngữ Lớn (LLM) để tích hợp bối cảnh mục lục/tiêu đề chương của báo cáo phát triển bền vững doanh nghiệp.

---
## 9. Tài liệu tham khảo (References — 16 công trình khoa học đã kiểm chứng) & Phiên bản Bài báo LaTeX

Toàn bộ 16 tài liệu tham khảo dưới đây đã được đối chiếu trực tiếp từ các nguồn xuất bản chính thức (ACL Anthology, PMLR, NeurIPS, JAIR, OpenReview, arXiv, Springer/Wiley DOI), sắp xếp đúng thứ tự trích dẫn chuẩn **IEEE Transactions** và đồng bộ 100% với bản thảo LaTeX tại [../latex_paper/main.tex](../latex_paper/main.tex) (gói Overleaf: [../latex_paper/Vi_ESG_Bert_Overleaf_Paper.zip](../latex_paper/Vi_ESG_Bert_Overleaf_Paper.zip)):

| STT | Trích dẫn đầy đủ (chuẩn IEEE) | Liên kết chính thức | Vai trò trong nghiên cứu |
| :-: | :--- | :--- | :--- |
| **[1]** | A. H. Huang, H. Wang, and Y. Yang, "FinBERT: A large language model for extracting information from financial text," *Contemporary Accounting Research*, vol. 40, no. 2, pp. 806–841, 2023. | [DOI Wiley](https://doi.org/10.1111/1911-3846.12832) | Công trình chuẩn mực cho bài toán trích xuất văn bản tài chính và phân loại 4 nhãn ESG (`finbert-esg`). |
| **[2]** | C. Guo, G. Pleiss, Y. Sun, and K. Q. Weinberger, "On calibration of modern neural networks," in *Proc. 34th Int. Conf. Mach. Learn. (ICML)*, PMLR, vol. 70, 2017, pp. 1321–1330. | [PMLR](https://proceedings.mlr.press/v70/guo17a.html) | Cơ sở công thức Expected Calibration Error (ECE), Reliability Diagram và phân tích hiện tượng quá tự tin. |
| **[3]** | J. Devlin, M.-W. Chang, K. Lee, and K. Toutanova, "BERT: Pre-training of deep bidirectional transformers for language understanding," in *Proc. NAACL-HLT*, 2019, pp. 4171–4186. | [ACL Anthology](https://aclanthology.org/N19-1423/) | Kiến trúc gốc của mô hình `google-bert/bert-base-multilingual-cased` (mBERT-cased). |
| **[4]** | V. Sanh, L. Debut, J. Chaumond, and T. Wolf, "DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter," *NeurIPS EMC² Workshop*, arXiv:1910.01108, 2019. | [arXiv:1910.01108](https://arxiv.org/abs/1910.01108) | Kiến trúc gốc của `distilbert-base-multilingual-cased` (mô hình đơn lẻ dẫn đầu Run 3 và mốc Pareto tốc độ). |
| **[5]** | Y. Liu et al., "RoBERTa: A robustly optimized BERT pretraining approach," *arXiv preprint arXiv:1907.11692*, 2019. | [arXiv:1907.11692](https://arxiv.org/abs/1907.11692) | Kiến trúc gốc của `FacebookAI/roberta-base` trong tập thể 7 mô hình. |
| **[6]** | D. Q. Nguyen and A. T. Nguyen, "PhoBERT: Pre-trained language models for Vietnamese," in *Findings of EMNLP*, 2020, pp. 1037–1042. | [ACL Anthology](https://aclanthology.org/2020.findings-emnlp.92/) | Kiến trúc gốc của `vinai/phobert-base` (cơ sở tách từ `pyvi` và mốc Baseline mạnh nhất). |
| **[7]** | K. Clark, M.-T. Luong, Q. V. Le, and C. D. Manning, "ELECTRA: Pre-training text encoders as discriminators rather than generators," in *Proc. ICLR*, 2020. | [OpenReview](https://openreview.net/forum?id=r1xMH1rtPB) | Cơ chế Replaced Token Detection làm nền tảng cho `NlpHUST/electra-base-vn` (Vi-ELECTRA). |
| **[8]** | N. Nguyen, T. Phan, D.-V. Nguyen, and K. V. Nguyen, "ViSoBERT: A pre-trained language model for Vietnamese social media text processing," in *Proc. EMNLP*, 2023, pp. 5191–5207. | [ACL Anthology](https://aclanthology.org/2023.emnlp-main.315/) | Kiến trúc gốc của mô hình `uitnlp/visobert` (tăng `+4.56%` so với Baseline sau Pseudo-Labeling). |
| **[9]** | C. D. Tran, N. H. Pham, A. T. Nguyen, T. S. Hy, and T. Vu, "ViDeBERTa: A powerful pre-trained language model for Vietnamese," in *Findings of EACL*, 2023, pp. 1071–1078. | [ACL Anthology](https://aclanthology.org/2023.findings-eacl.79/) | Kiến trúc gốc của mô hình `Fsoft-AIC/videberta-base` trong tập thể 7 mô hình. |
| **[10]** | T. G. Dietterich, "Ensemble methods in machine learning," in *Proc. Int. Workshop Multiple Classifier Systems (MCS)*, LNCS, vol. 1857, Springer, 2000, pp. 1–15. | [DOI Springer](https://doi.org/10.1007/3-540-45014-9_1) | Cơ sở lý thuyết về tính đa dạng kiến trúc (*ensemble diversity*) cho hệ thống 7 mô hình. |
| **[11]** | D.-H. Lee, "Pseudo-label: The simple and efficient semi-supervised learning method for deep neural networks," in *ICML Workshop on Challenges in Representation Learning (WREPL)*, 2013. | [Semantic Scholar](https://www.semanticscholar.org/paper/Pseudo-Label-%3A-The-Simple-and-Efficient-Method-for-Lee/798d9840d2439a0e5d47bcf5d164aa46d5e7dc26) | Nền tảng phương pháp học bán giám sát bằng nhãn giả (*Pseudo-Labeling*). |
| **[12]** | Y. Grandvalet and Y. Bengio, "Semi-supervised learning by entropy minimization," in *Advances in Neural Information Processing Systems (NeurIPS)*, vol. 17, 2004, pp. 529–536. | [NeurIPS](https://proceedings.neurips.cc/paper/2004/hash/9b086e3f45f3d73bda8be09689454aa3-Abstract.html) | Cơ sở toán học tối thiểu hóa entropy cho việc tuyển chọn nhãn giả độ tin cậy cao / entropy thấp. |
| **[13]** | Q. McNemar, "Note on the sampling error of the difference between correlated proportions or percentages," *Psychometrika*, vol. 12, no. 2, pp. 153–157, 1947. | [DOI Springer](https://doi.org/10.1007/BF02295996) | Kiểm định thống kê McNemar ($\chi^2$) xác minh ý nghĩa thống kê của Final Ensemble. |
| **[14]** | J. R. Landis and G. G. Koch, "The measurement of observer agreement for categorical data," *Biometrics*, vol. 33, no. 1, pp. 159–174, 1977. | [DOI Biometrics](https://doi.org/10.2307/2529310) | Thang đánh giá hệ số đồng thuận Cohen's Kappa cho Ma trận đồng thuận cặp giữa 7 mô hình. |
| **[15]** | G. Hinton, O. Vinyals, and J. Dean, "Distilling the knowledge in a neural network," *arXiv preprint arXiv:1503.02531*, 2015. | [arXiv:1503.02531](https://arxiv.org/abs/1503.02531) | Cơ sở cho hướng phát triển chưng cất tri thức từ Ensemble 7M sang một mô hình gọn nhẹ. |
| **[16]** | C. Northcutt, L. Jiang, and I. Chuang, "Confident learning: Estimating uncertainty in dataset labels," *Journal of Artificial Intelligence Research (JAIR)*, vol. 70, pp. 1373–1411, 2021. | [JAIR Official](https://jair.org/index.php/jair/article/view/12125) | Cơ sở cho hướng phát triển rà soát nhiễu nhãn nền (Cleanlab) phát hiện trong phần Phân tích lỗi. |